In [2]:
# ── STEP 4: A* ROUTING ────────────────────────────────────
# New file: astar_routing.py (or new notebook cell)

import osmnx as ox
import heapq
import json
import numpy as np
from math import radians, sin, cos, sqrt, atan2

# ── LOAD GRAPH ────────────────────────────────────────────
print("Loading graph...")
G = ox.load_graphml('chicago_safety_graph.graphml')
print(f"✅ Loaded: {G.number_of_nodes():,} nodes, "
      f"{G.number_of_edges():,} edges")

Loading graph...
✅ Loaded: 304,603 nodes, 1,022,766 edges


In [23]:
# ── HELPER FUNCTIONS ──────────────────────────────────────
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1, phi2 = radians(lat1), radians(lat2)
    dphi       = radians(lat2 - lat1)
    dlambda    = radians(lon2 - lon1)
    a = sin(dphi/2)**2 + cos(phi1)*cos(phi2)*sin(dlambda/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))


# ── FIX: STRONGER TIME MULTIPLIER + HIGHER PENALTY ────────

def get_time_multiplier(hour):
    """
    Much more aggressive tiers so night routing
    actually forces A* to pick different paths
    """
    if 6 <= hour <= 17:    return 1.0   # daytime
    elif 18 <= hour <= 21: return 3.0   # evening
    elif 22 <= hour <= 23: return 6.0   # late night
    else:                  return 10.0  # 0-5 AM — extreme


def compute_edge_cost(data, current_hour, safety_weight):
    length       = float(data.get('length', 50))
    danger_score = float(data.get('danger_score', 0.0))
    time_mult    = get_time_multiplier(current_hour)

    # Penalty now: danger=0.7, night(10x), weight=1.0
    # → 0.7 × 500 × 10 = 3500m equivalent detour
    # Safe path at 2000m extra will always win
    danger_penalty = (danger_score ** 1.5) * 500 * time_mult * safety_weight
    #                 ↑ squared to punish high-danger edges extra hard

    return max(1.0, length + danger_penalty)

In [24]:
# ── A* ALGORITHM ──────────────────────────────────────────
def astar(G, start_node, end_node, current_hour, safety_weight):
    end_lat = G.nodes[end_node]['y']
    end_lon = G.nodes[end_node]['x']

    def heuristic(node):
        return haversine(G.nodes[node]['y'], G.nodes[node]['x'],
                         end_lat, end_lon)

    open_set  = []
    came_from = {}
    g_score   = {start_node: 0.0}
    heapq.heappush(open_set, (heuristic(start_node), start_node))
    visited   = set()

    while open_set:
        _, current = heapq.heappop(open_set)

        if current == end_node:
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start_node)
            path.reverse()
            return path

        if current in visited:
            continue
        visited.add(current)

        for neighbor in G.neighbors(current):
            if neighbor in visited:
                continue
            edge_data = min(
                G[current][neighbor].values(),
                key=lambda d: compute_edge_cost(d, current_hour,
                                                safety_weight)
            )
            tentative_g = (g_score[current] +
                           compute_edge_cost(edge_data, current_hour,
                                             safety_weight))

            if tentative_g < g_score.get(neighbor, float('inf')):
                came_from[neighbor] = current
                g_score[neighbor]   = tentative_g
                f                   = tentative_g + heuristic(neighbor)
                heapq.heappush(open_set, (f, neighbor))

    return None

In [26]:
# ── SAFETY SCORE ──────────────────────────────────────────
# ── FIX SAFETY SCORE FORMULA ─────────────────────────────
def calculate_safety_score(G, path_nodes, current_hour):
    if not path_nodes or len(path_nodes) < 2:
        return 50.0

    dangers = []
    total_length = 0.0

    for i in range(len(path_nodes) - 1):
        u, v = path_nodes[i], path_nodes[i+1]
        if G.has_edge(u, v):
            edge = min(G[u][v].values(),
                       key=lambda d: float(d.get('length', 50)))
            dangers.append(float(edge.get('danger_score', 0.0)))
            total_length += float(edge.get('length', 50))

    if not dangers:
        return 50.0

    avg_danger = np.mean(dangers)
    time_mult  = get_time_multiplier(current_hour)

    # OLD (broken): 100 - (avg_danger * 60 * time_mult)
    # At night: 0.5 * 60 * 10 = 300 → always 0
    
    # NEW: separate danger and time components
    # danger_component: 0-70 points deducted based on route danger
    # time_component:   0-30 points deducted based on hour
    danger_component = avg_danger * 70          # max 70 pts deducted
    time_component   = ((time_mult - 1) / 9) * 30  # 0-30 pts at night
    
    score = 100 - danger_component - time_component
    return round(max(0.0, min(100.0, score)), 1)

In [29]:
# ── MAIN ROUTING FUNCTION ─────────────────────────────────
def find_all_routes(start_lat, start_lon,
                    end_lat,   end_lon, current_hour):
    start_node = ox.nearest_nodes(G, X=start_lon, Y=start_lat)
    end_node   = ox.nearest_nodes(G, X=end_lon,   Y=end_lat)

    print(f"Hour: {current_hour}:00 | "
          f"Multiplier: {get_time_multiplier(current_hour)}x")

    results = {}
    configs = [
        ('fastest',  0.0),
        ('balanced', 0.5),
        ('safest',   1.0),
    ]

    for name, weight in configs:
        print(f"  Running {name}...", end=' ')
        path = astar(G, start_node, end_node, current_hour, weight)

        if path is None:
            print("❌ no path found")
            results[name] = None
            continue

        coords = [(G.nodes[n]['y'], G.nodes[n]['x']) for n in path]
        total_len = sum(
            float(min(G[path[i]][path[i+1]].values(),
                  key=lambda d: float(d.get('length', 50))
                  ).get('length', 50))
            for i in range(len(path)-1)
            if G.has_edge(path[i], path[i+1])
        )
        score = calculate_safety_score(G, path, current_hour)
        results[name] = {
            'coordinates':    coords,
            'node_ids':       path,
            'safety_score':   score,
            'total_length_m': round(total_len, 1)
        }
        print(f"✅ {len(path)} nodes | "
              f"{total_len:.0f}m | Safety: {score}/100")

    return results

In [30]:
# ── FINAL CLEAN TEST ──────────────────────────────────────
# Use Loop → Wicker Park (proven to diverge)

print("=" * 55)
print("FINAL VALIDATION: Loop → Wicker Park")
print("=" * 55)

test_cases = [
    (2,  "2 AM  — Night 🌙 (10x multiplier)"),
    (12, "12 PM — Day ☀️  (1x multiplier)"),
    (21, "9 PM  — Evening 🌆 (6x multiplier)"),
]

all_results = {}
for hour, label in test_cases:
    print(f"\n{label}")
    r = find_all_routes(
        start_lat=41.8827, start_lon=-87.6233,
        end_lat=41.9100,   end_lon=-87.6770,
        current_hour=hour
    )
    all_results[hour] = r

    for rtype in ['fastest', 'balanced', 'safest']:
        if r.get(rtype):
            print(f"  {rtype:8}: {r[rtype]['total_length_m']:7.0f}m "
                  f"| Safety: {r[rtype]['safety_score']:5.1f}/100")

FINAL VALIDATION: Loop → Wicker Park

2 AM  — Night 🌙 (10x multiplier)
Hour: 2:00 | Multiplier: 10.0x
  Running fastest... ✅ 166 nodes | 6076m | Safety: 20.3/100
  Running balanced... ✅ 247 nodes | 13225m | Safety: 50.6/100
  Running safest... ✅ 247 nodes | 13266m | Safety: 50.6/100
  fastest :    6076m | Safety:  20.3/100
  balanced:   13225m | Safety:  50.6/100
  safest  :   13266m | Safety:  50.6/100

12 PM — Day ☀️  (1x multiplier)
Hour: 12:00 | Multiplier: 1.0x
  Running fastest... ✅ 166 nodes | 6076m | Safety: 50.3/100
  Running balanced... ✅ 120 nodes | 6808m | Safety: 51.7/100
  Running safest... ✅ 234 nodes | 12236m | Safety: 78.7/100
  fastest :    6076m | Safety:  50.3/100
  balanced:    6808m | Safety:  51.7/100
  safest  :   12236m | Safety:  78.7/100

9 PM  — Evening 🌆 (6x multiplier)
Hour: 21:00 | Multiplier: 3.0x
  Running fastest... ✅ 166 nodes | 6076m | Safety: 43.6/100
  Running balanced... ✅ 247 nodes | 13225m | Safety: 74.0/100
  Running safest... ✅ 247 nodes | 132

In [32]:
# ── SHOW THE KEY COMPARISON ───────────────────────────────
print("\n" + "=" * 55)
print("KEY COMPARISON: Does safest route get longer at night?")
print("=" * 55)

night = all_results[2]
day   = all_results[12]

if night.get('safest') and day.get('safest'):
    n_len = night['safest']['total_length_m']
    d_len = day['safest']['total_length_m']
    n_sc  = night['safest']['safety_score']
    d_sc  = day['safest']['safety_score']
    extra = n_len - d_len

    print(f"\nSAFEST route:")
    print(f"  Night (2AM):  {n_len:.0f}m | Score: {n_sc}/100")
    print(f"  Day  (12PM):  {d_len:.0f}m | Score: {d_sc}/100")
    print(f"  Extra detour at night: {extra:.0f}m")
    print(f"\n  Verdict: ", end="")
    if extra > 500 and n_sc < d_sc:
        print("✅ PASS — algorithm correctly avoids danger at night")
    elif extra > 0:
        print("⚠️  PARTIAL — routes differ but detour is small")
    else:
        print("❌ FAIL — same route day and night")

if night.get('fastest') and night.get('safest'):
    f_len = night['fastest']['total_length_m']
    s_len = night['safest']['total_length_m']
    print(f"\nAt 2AM — Fastest vs Safest comparison:")
    print(f"  Fastest: {f_len:.0f}m | Score: "
          f"{night['fastest']['safety_score']}/100")
    print(f"  Safest:  {s_len:.0f}m | Score: "
          f"{night['safest']['safety_score']}/100")
    print(f"  Safety tradeoff: +{s_len-f_len:.0f}m for "
          f"+{night['safest']['safety_score']-night['fastest']['safety_score']:.1f} "
          f"safety points")


KEY COMPARISON: Does safest route get longer at night?

SAFEST route:
  Night (2AM):  13266m | Score: 50.6/100
  Day  (12PM):  12236m | Score: 78.7/100
  Extra detour at night: 1030m

  Verdict: ✅ PASS — algorithm correctly avoids danger at night

At 2AM — Fastest vs Safest comparison:
  Fastest: 6076m | Score: 20.3/100
  Safest:  13266m | Score: 50.6/100
  Safety tradeoff: +7189m for +30.3 safety points


In [33]:
# ── SAVE FINAL ROUTES FOR BACKEND ────────────────────────
import json

def routes_to_json(routes):
    out = {}
    for name, r in routes.items():
        if r:
            out[name] = {
                'coordinates':    r['coordinates'],
                'safety_score':   r['safety_score'],
                'total_length_m': r['total_length_m']
            }
    return out

with open('test_routes_2am.json',  'w') as f:
    json.dump(routes_to_json(all_results[2]),  f, indent=2)
with open('test_routes_12pm.json', 'w') as f:
    json.dump(routes_to_json(all_results[12]), f, indent=2)
with open('test_routes_9pm.json',  'w') as f:
    json.dump(routes_to_json(all_results[21]), f, indent=2)

print("\n✅ Saved route JSONs for all 3 time periods")
print("✅ Step 4 Complete — Ready for Step 5: FastAPI Backend")


✅ Saved route JSONs for all 3 time periods
✅ Step 4 Complete — Ready for Step 5: FastAPI Backend
